## Code to explore, filter and sample cells

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import cellxgene_census

import os
from pathlib import Path


In [ ]:
# path to cwd

path = '/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE'

In [3]:
# other paths
data_dir = os.path.join(path, 'data')
os.makedirs(data_dir, exist_ok = True)

results_dir = os.path.join(data_dir, 'results')
os.makedirs(results_dir, exist_ok = True)

# print('Working directory:', results_dir)

In [4]:
# Immune Human Atlas paths

sample_id_path = os.path.join(data_dir, 'iha_sample_ids.csv')
raw_data_path = os.path.join(data_dir, 'iha_expression_raw.h5ad')
prepared_data_path = os.path.join(data_dir, 'iha_expression_prep.h5ad')

In [5]:
census_version = '2025-11-08'
collection_id = '77f9d7e9-5675-49c3-abed-ce02f39eef1b'

# Sampling
num_donors_per_group = 8
min_cells_don = 2000
min_types_don = 8
min_per_type = 10
max_per_type = 100

# Immune Human Atlas dataset ID in Census datadase
main_id = 'e522d2cd-7927-4e59-a4ed-064009569279' 

In [6]:
with cellxgene_census.open_soma(census_version = census_version) as census:
    datasets = (census['census_info']['datasets'].read().concat().to_pandas())

atlas_datasets = datasets[datasets['collection_id'] == collection_id].copy()

assert main_id in atlas_datasets["dataset_id"].values

display(atlas_datasets[atlas_datasets["dataset_id"] == main_id][["dataset_id", "dataset_title", "dataset_total_cell_count"]])

,dataset_id,dataset_title,dataset_total_cell_count
1797,e522d2cd-7927-4e59-a4ed-064009569279,Human Immune Health Atlas,1821725


In [7]:
obs_cols = ['soma_joinid',
            'dataset_id',
            'donor_id',
            'cell_type',
            'disease',
            'sex',
            'development_stage',
            'tissue',
            'tissue_general',
            'assay',
            'is_primary_data']

with cellxgene_census.open_soma(census_version = census_version) as census:
    obs = census['census_data']['homo_sapiens'].obs.read(value_filter = f'dataset_id == "{main_id}"',
                                                         column_names = obs_cols)
    obs = obs.concat().to_pandas()

display(obs.head())

print('Cells:', len(obs))
print('Donors:', obs.donor_id.nunique())
print('Cell types:', obs.cell_type.nunique())

,soma_joinid,dataset_id,donor_id,cell_type,disease,sex,development_stage,tissue,tissue_general,assay,is_primary_data
0,103774024,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,mucosal invariant T cell,normal,female,32-year-old stage,blood,blood,10x 3' v3,False
1,103774025,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"naive thymus-derived CD4-positive, alpha-beta ...",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
2,103774026,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"CD14-positive, CD16-negative classical monocyte",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
3,103774027,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"central memory CD4-positive, alpha-beta T cell",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
4,103774028,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"CD14-positive, CD16-negative classical monocyte",normal,female,32-year-old stage,blood,blood,10x 3' v3,False


Cells: 1821725
Donors: 108
Cell types: 38


In [8]:
obs_cols = ['soma_joinid',
            'dataset_id',
            'donor_id',
            'cell_type',
            'disease',
            'sex',
            'development_stage',
            'tissue',
            'tissue_general',
            'assay',
            'is_primary_data']

with cellxgene_census.open_soma(census_version = census_version) as census:
    obs = (census['census_data']['homo_sapiens'].obs.read(value_filter = f'dataset_id == "{main_id}"',
                                                          column_names = obs_cols).concat().to_pandas())

print('Cells:', len(obs))
print('Donors:', obs.donor_id.nunique())
print('Cell types:', obs.cell_type.nunique())

display(obs.head())

Cells: 1821725
Donors: 108
Cell types: 38


,soma_joinid,dataset_id,donor_id,cell_type,disease,sex,development_stage,tissue,tissue_general,assay,is_primary_data
0,103774024,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,mucosal invariant T cell,normal,female,32-year-old stage,blood,blood,10x 3' v3,False
1,103774025,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"naive thymus-derived CD4-positive, alpha-beta ...",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
2,103774026,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"CD14-positive, CD16-negative classical monocyte",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
3,103774027,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"central memory CD4-positive, alpha-beta T cell",normal,female,32-year-old stage,blood,blood,10x 3' v3,False
4,103774028,e522d2cd-7927-4e59-a4ed-064009569279,BR1001,"CD14-positive, CD16-negative classical monocyte",normal,female,32-year-old stage,blood,blood,10x 3' v3,False


## Clinical metadata and donor information

In [9]:
clinical = pd.read_csv(f'{path}/human_immune_health_atlas_metadata_clinical_labs.csv')

# print('Clinical metadata columns:')
# for col in clinical.columns:
#     print(col)

In [10]:
clinical_cols = ['subject.subjectGuid', 'subject.ageAtFirstDraw', 'sample.sampleKitGuid', 'sample.subjectAgeAtDraw']

display(clinical[clinical_cols].head(10))

,subject.subjectGuid,subject.ageAtFirstDraw,sample.sampleKitGuid,sample.subjectAgeAtDraw
0,BR1001,32,KT00001,32
1,BR1002,28,KT00002,28
2,BR1003,30,KT00003,30
3,BR1004,30,KT00004,30
4,BR1005,27,KT00006,27
5,BR1006,31,KT00010,31
6,BR1008,34,KT00012,34
7,BR1007,30,KT00014,30
8,BR2003,59,KT00015,59
9,BR2002,57,KT00016,57


In [11]:
assert clinical['subject.subjectGuid'].is_unique

clinical_renamed = clinical.rename(columns = {'subject.subjectGuid': 'donor_id', # to merge with obs
                                              'subject.ageAtFirstDraw': 'age_at_first_draw',
                                              'sample.subjectAgeAtDraw': 'age'}).copy()

obs = obs.merge(clinical_renamed, on = 'donor_id', how = 'left', validate = 'many_to_one')

assert obs["age"].notna().all()

display(obs[["donor_id", "age_at_first_draw", "age"]].drop_duplicates().head(10))

,donor_id,age_at_first_draw,age
0,BR1001,32,32
18527,BR1002,28,28
37484,BR1003,30,30
56486,BR1004,30,30
73836,BR1005,27,27
91524,BR1006,31,31
108842,BR1008,34,34
124747,BR1007,30,30
141843,BR2003,59,59
157433,BR2002,57,57


In [12]:
# Check disease annotations per donor
disease_per_donor = (obs.groupby('donor_id')['disease'].nunique())
assert disease_per_donor.max() == 1, ('Some donors have multiple disease annotations!')

# Keep healthy cells only
obs_healthy = obs[obs['disease'] == 'normal'].copy()


# Define age groups

age_labels = ['< 18', '18 - 45', '> 45']
def assign_age_group(age):
    if age < 18:
        return age_labels[0]
    elif age <= 45:
        return age_labels[1]
    else:
        return age_labels[2]


obs_healthy['age_group'] = obs_healthy['age'].apply(assign_age_group)
obs_healthy['age_group'] = pd.Categorical(obs_healthy['age_group'], categories = age_labels, ordered = True)


# Age distribution among healthy donors
healthy_donor_age = obs_healthy[['donor_id', 'age', 'age_group']].drop_duplicates()

display(healthy_donor_age.groupby('age_group', observed = False).agg(donors = ('donor_id', 'nunique'), 
                                                                   min_age = ('age', 'min'),
                                                                   max_age = ('age', 'max')))


,donors,min_age,max_age
age_group,,,
< 18,13,11,12
18 - 45,29,25,35
> 45,21,55,65


In [13]:
# Cell and cell-type distribution per donor

donor_stats = (obs_healthy.groupby(["donor_id", "age_group"], observed = True).agg(total_cells = ("soma_joinid", "size"),
                                                                                   cell_types = ("cell_type", "nunique")).reset_index())

display(donor_stats.sort_values(["age_group", "total_cells"], ascending = [True, False]))

,donor_id,age_group,total_cells,cell_types
50,UP1008,< 18,20902,37
53,UP1014,< 18,20613,37
54,UP1015,< 18,18620,37
62,UP1034,< 18,18018,37
60,UP1031,< 18,17706,38
...,...,...,...,...
48,BR2046,> 45,14290,37
41,BR2032,> 45,14161,38
32,BR2015,> 45,13527,37
46,BR2040,> 45,13397,37


In [14]:
# Sample patients with a necessary amount of cells
eligible_donors = donor_stats[(donor_stats["total_cells"] >= min_cells_don) &
                              (donor_stats["cell_types"] >= min_types_don)].copy()

eligibility_summary = (eligible_donors.groupby("age_group", observed=False).agg(eligible_donors=("donor_id", "nunique")))

display(eligibility_summary)

,eligible_donors
age_group,
< 18,13
18 - 45,29
> 45,21


In [15]:
# Estimate cell counts after donor × cell type filtering

donor_sampling_stats = obs_healthy.groupby(["donor_id", "age_group", "cell_type"], observed=True).size().reset_index(name="n_cells")

# Remove donor × cell type groups with fewer than 10 cells
donor_sampling_stats = donor_sampling_stats[donor_sampling_stats["n_cells"] >= min_per_type].copy()

# Apply maximum of 100 cells per donor × cell type
donor_sampling_stats["sampled_cells"] = donor_sampling_stats["n_cells"].clip(upper = max_per_type)

# Summarize expected sampling results per donor
donor_sampling_summary = donor_sampling_stats.groupby(["donor_id", "age_group"], observed = True).agg(cell_types = ("cell_type", "nunique"),
                                                                                                      original_cells = ("n_cells", "sum"),
                                                                                                      expected_sampled_cells = ("sampled_cells", "sum")).reset_index()

# Another constraint of min. 2000 cels _sampled_ per donor
donor_sampling_summary = donor_sampling_summary[donor_sampling_summary["expected_sampled_cells"] >= min_cells_don]


display(donor_sampling_summary.sort_values(["age_group", "expected_sampled_cells"], ascending = [True, False]))

,donor_id,age_group,cell_types,original_cells,expected_sampled_cells
57,UP1023,< 18,30,13250,2578
53,UP1014,< 18,31,20583,2533
50,UP1008,< 18,32,20883,2392
55,UP1017,< 18,29,16816,2349
54,UP1015,< 18,31,18590,2312
56,UP1021,< 18,29,14088,2286
52,UP1012,< 18,26,16119,2275
60,UP1031,< 18,29,17677,2263
51,UP1011,< 18,29,14610,2222
61,UP1032,< 18,30,16304,2208


In [16]:
display(donor_sampling_summary.groupby("age_group", observed = False).agg(donors = ("donor_id", "nunique"),
                                                                          min_cells = ("expected_sampled_cells", "min"),
                                                                          median_cells = ("expected_sampled_cells", "median"),
                                                                          max_cells = ("expected_sampled_cells", "max"),
                                                                          min_cell_types = ("cell_types", "min")))

,donors,min_cells,median_cells,max_cells,min_cell_types
age_group,,,,,
< 18,12,2116,2280.5,2578,26
18 - 45,29,2093,2246.0,2520,26
> 45,19,2119,2236.0,2438,27


In [17]:
# Select 8 donors per age group
selected_donors = donor_sampling_summary.groupby("age_group", observed = False, group_keys = False).sample(n = num_donors_per_group, random_state = 42).reset_index(drop = True)

display(selected_donors[["donor_id", "age_group", "expected_sampled_cells", "cell_types"]])

,donor_id,age_group,expected_sampled_cells,cell_types
0,UP1032,< 18,2208,30
1,UP1031,< 18,2263,29
2,UP1008,< 18,2392,32
3,UP1025,< 18,2133,31
4,UP1017,< 18,2349,29
5,UP1012,< 18,2275,26
6,UP1011,< 18,2222,29
7,UP1034,< 18,2116,28
8,BR1054,18 - 45,2107,30
9,BR1021,18 - 45,2126,30


## Cell sampling

Selection of up to 100 cells per donor × cell type. Groups containing fewer than 10 cells are excluded.

In [18]:
selected_donor_ids = selected_donors["donor_id"].tolist()

sampling_obs = obs_healthy[obs_healthy["donor_id"].isin(selected_donor_ids)].copy()

In [19]:
# Sample cells per donor × cell type

sampled_groups = []

for (donor, cell_type), group in sampling_obs.groupby(["donor_id", "cell_type"], observed = True):
    if len(group) < min_per_type:
        continue
    n = min(len(group), max_per_type)
    sampled_group = group.sample(n = n, random_state = 42)
    sampled_groups.append(sampled_group)

sampled_obs = pd.concat(sampled_groups, ignore_index = True)

print("Total sampled cells:", len(sampled_obs))

Total sampled cells: 54000


In [20]:
# Verify sampling constraints

sampling_check = sampled_obs.groupby(["donor_id", "cell_type"], observed=True).size()

assert sampling_check.min() >= min_per_type
assert sampling_check.max() <= max_per_type

assert sampled_obs["donor_id"].nunique() == 24

assert sampled_obs["soma_joinid"].is_unique

print("All sampling constraints satisfied.")

All sampling constraints satisfied.


In [21]:
# Final sampling summary

final_summary = sampled_obs.groupby(["donor_id", "age_group"], observed=True).agg(cells = ("soma_joinid", "size"),
                                                                                  cell_types = ("cell_type", "nunique")).reset_index()

display(final_summary.groupby("age_group", observed = True).agg(donors = ("donor_id", "nunique"),
                                                              total_cells = ("cells", "sum"),
                                                              min_cells = ("cells", "min"),
                                                              median_cells = ("cells", "median"),
                                                              max_cells = ("cells", "max"),
                                                              min_cell_types = ("cell_types", "min")))

,donors,total_cells,min_cells,median_cells,max_cells,min_cell_types
age_group,,,,,,
< 18,8,17958,2116,2242.5,2392,26
18 - 45,8,17979,2107,2236.5,2407,30
> 45,8,18063,2119,2232.0,2438,27


In [22]:
# Save sampled cell IDs and metadata

sample_ids = sampled_obs[["soma_joinid", "donor_id", "cell_type", "age", "age_group", "sex"]].copy()

sample_ids.to_csv(sample_id_path, index = False)